# Goal: statistical-summary 

In [8]:
""" 
Goal: statistical-summary
Author: Rudra Prasad Bhuyan
"""

' \nGoal: statistical-summary\nAuthor: Rudra Prasad Bhuyan\n'

In [9]:
import polars as pl

In [10]:
path = r"C:\Users\Rudra\Desktop\rural-financial-inclusion-govt-scheme-recommendation\parquet-data\lev-13\data\lev-13_merged.parquet"
pdf = pl.scan_parquet(path)

In [11]:
pdf.collect_schema()

Schema([('Survey_Name', String),
        ('Year', String),
        ('FSU_Serial_No', String),
        ('Sector', String),
        ('State', String),
        ('NSS_Region', String),
        ('District', String),
        ('Stratum', String),
        ('Sub_stratum', String),
        ('Panel', String),
        ('Sub_sample', String),
        ('FOD_Sub_Region', String),
        ('Sample_SU_No', String),
        ('Sample_Sub_Division_No', String),
        ('Second_Stage_Stratum_No', String),
        ('Sample_Household_No', String),
        ('Questionnaire_No', String),
        ('Level', String),
        ('ITEM_CODE', String),
        ('FIRST_PURCHASE_NUMBER', Float64),
        ('PURCHASED_ON_HIRE', String),
        ('FIRST_PURCHASE_VALUE', Float64),
        ('REPAIR_COST', Float64),
        ('SECOND_HAND_NUMBER', Float64),
        ('SECOND_HAND_VALUE', Float64),
        ('TOTAL_EXPENDITURE', Int64),
        ('MULTIPLIER', Int64)])

# Useful Variables

In [20]:
cols = [
    'ITEM_CODE',
'FIRST_PURCHASE_NUMBER',
'PURCHASED_ON_HIRE',
'FIRST_PURCHASE_VALUE',
'REPAIR_COST',
'TOTAL_EXPENDITURE',
'SECOND_HAND_NUMBER',
'SECOND_HAND_VALUE',
]

In [21]:
df = pdf.select(cols)

In [22]:
df.head(2).collect()

ITEM_CODE,FIRST_PURCHASE_NUMBER,PURCHASED_ON_HIRE,FIRST_PURCHASE_VALUE,REPAIR_COST,TOTAL_EXPENDITURE,SECOND_HAND_NUMBER,SECOND_HAND_VALUE
str,f64,str,f64,f64,i64,f64,f64
"""442""",2.0,"""""",300.0,null,300,null,null
"""552""",null,"""""",null,400.0,400,null,null


In [23]:
df = df.with_columns(
    [pl.col(col).cast(pl.Int32, strict=False) for col in cols]
)

In [24]:
unique_counts = df.select(pl.all().n_unique()).collect()
unique_counts

ITEM_CODE,FIRST_PURCHASE_NUMBER,PURCHASED_ON_HIRE,FIRST_PURCHASE_VALUE,REPAIR_COST,TOTAL_EXPENDITURE,SECOND_HAND_NUMBER,SECOND_HAND_VALUE
u32,u32,u32,u32,u32,u32,u32,u32
78,34,3,13900,6320,16957,6,367


# Logic

In [25]:
categorical_cols = []
numerical_cols = []

for col in df.columns:
    if unique_counts[col][0] <13:
        categorical_cols.append(col)
    else:
        numerical_cols.append(col)


C:\Users\Rudra\AppData\Local\Temp\ipykernel_7440\2508435801.py:4: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  for col in df.columns:


# Numerical Columns

In [26]:
stats = df.select(numerical_cols).describe()
with pl.Config(tbl_rows=-1, tbl_cols=-1):
    display(stats.to_pandas().T)

,0,1,2,3,4,5,6,7,8
statistic,count,null_count,mean,std,min,25%,50%,75%,max
ITEM_CODE,9903498.0,0.0,540.068718,158.44473,35.0,559.0,587.0,625.0,649.0
FIRST_PURCHASE_NUMBER,1690062.0,8213436.0,1.801187,1.489083,0.0,1.0,1.0,2.0,40.0
FIRST_PURCHASE_VALUE,6901922.0,3001576.0,1793.900681,12652.506766,0.0,300.0,600.0,1350.0,4501200.0
REPAIR_COST,3997882.0,5905616.0,1507.810618,7717.943048,0.0,210.0,500.0,1500.0,1680000.0
TOTAL_EXPENDITURE,9903498.0,0.0,1879.647235,11836.528719,1.0,300.0,620.0,1500.0,4511750.0
SECOND_HAND_VALUE,18794.0,9884704.0,10943.445993,34752.11268,0.0,950.0,3000.0,6500.0,620000.0


# Categorical Columns

In [27]:
for col in categorical_cols:
    print(col)
    counts = df.select(pl.col(col).value_counts(sort=True)).collect()
    
    with pl.Config(tbl_rows=-1, tbl_cols=-1):
        display(counts.unnest(col))

PURCHASED_ON_HIRE


PURCHASED_ON_HIRE,count
i32,u32
null,6827686
2,3045072
1,30740


SECOND_HAND_NUMBER


SECOND_HAND_NUMBER,count
i32,u32
null,9895914
1,7170
2,370
3,22
4,16
0,6
